In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
#!/usr/bin/env python3

import argparse
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

from pathlib import Path

BASE_DATA_ROOT = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export")
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv")
OUTPUT = Path("drive/MyDrive/kaggle_cs3780_sp26/gpu_models_metrics.json")

IMG_HEIGHT = 32
IMG_WIDTH = 64
BATCH_SIZE = 256
EPOCHS = 20
LR_LOGREG = 1e-2
LR_CNN = 1e-3
NUM_WORKERS = 2
SEED = 2026


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.samples = []

        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("L")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("L")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class LogisticRegressionTorch(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.linear(x)


class SmallBirdCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 32x64 -> 16x32

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 16x32 -> 8x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 8x16 -> 4x8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def run_one_epoch(model, loader, criterion, optimizer, device, train: bool):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds)


def train_model(model, model_name, train_loader, test_loader, classes, device, epochs, lr):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    best_test_acc = -1.0
    best_y_true = None
    best_y_pred = None

    for epoch in range(1, epochs + 1):
        print(f"\n[{model_name}] Epoch {epoch}/{epochs}")
        train_loss, train_acc, _, _ = run_one_epoch(model, train_loader, criterion, optimizer, device, train=True)
        test_loss, test_acc, y_true, y_pred = run_one_epoch(model, test_loader, criterion, optimizer, device, train=False)

        print(f"[{model_name}] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
        print(f"[{model_name}] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

        history.append({
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_accuracy": float(train_acc),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_acc),
        })

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_y_true = y_true
            best_y_pred = y_pred

    report = classification_report(
        best_y_true,
        best_y_pred,
        target_names=classes,
        output_dict=True,
        zero_division=0,
    )

    final_train_loss, final_train_acc, _, _ = run_one_epoch(model, train_loader, criterion, optimizer, device, train=False)

    return {
        "final_train_accuracy": float(final_train_acc),
        "best_test_accuracy": float(best_test_acc),
        "history": history,
        "classification_report": report,
    }


def main():
    set_seed(SEED)

    train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
    test_dir = BASE_DATA_ROOT / TEST_SUBDIR
    solution_csv = SOLUTION_CSV

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    transform = transforms.Compose([
        transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
        transforms.ToTensor(),
    ])

    train_dataset = TrainFolderDataset(train_dir, transform=transform)
    test_dataset = TestWithSolutionDataset(
        test_dir=test_dir,
        solution_csv=solution_csv,
        class_to_idx=train_dataset.class_to_idx,
        transform=transform,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    num_classes = len(train_dataset.classes)
    input_dim = IMG_HEIGHT * IMG_WIDTH

    logreg_model = LogisticRegressionTorch(input_dim=input_dim, num_classes=num_classes)
    cnn_model = SmallBirdCNN(num_classes=num_classes)

    logreg_results = train_model(
        model=logreg_model,
        model_name="gpu_logistic_regression",
        train_loader=train_loader,
        test_loader=test_loader,
        classes=train_dataset.classes,
        device=device,
        epochs=EPOCHS,
        lr=LR_LOGREG,
    )

    cnn_results = train_model(
        model=cnn_model,
        model_name="gpu_cnn",
        train_loader=train_loader,
        test_loader=test_loader,
        classes=train_dataset.classes,
        device=device,
        epochs=EPOCHS,
        lr=LR_CNN,
    )

    results = {
        "config": {
            "train_dir": str(train_dir),
            "test_dir": str(test_dir),
            "solution_csv": str(solution_csv),
            "img_height": IMG_HEIGHT,
            "img_width": IMG_WIDTH,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr_logreg": LR_LOGREG,
            "lr_cnn": LR_CNN,
            "seed": SEED,
            "device": str(device),
        },
        "num_train": len(train_dataset),
        "num_test": len(test_dataset),
        "num_classes": num_classes,
        "classes": train_dataset.classes,
        "models": {
            "gpu_logistic_regression": logreg_results,
            "gpu_cnn": cnn_results,
        },
    }

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT.open("w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("\n==== Final Results ====")
    print(
        f"gpu_logistic_regression train={logreg_results['final_train_accuracy']:.4f}, "
        f"test={logreg_results['best_test_accuracy']:.4f}"
    )
    print(
        f"gpu_cnn train={cnn_results['final_train_accuracy']:.4f}, "
        f"test={cnn_results['best_test_accuracy']:.4f}"
    )
    print(f"Saved metrics to {OUTPUT}")


if __name__ == "__main__":
    main()

Using device: cuda
GPU: Tesla T4

[gpu_logistic_regression] Epoch 1/20


[gpu_logistic_regression] train_loss=3.6469, train_acc=0.1786
[gpu_logistic_regression] test_loss=2.6479, test_acc=0.1786

[gpu_logistic_regression] Epoch 2/20


[gpu_logistic_regression] train_loss=2.1240, train_acc=0.2309
[gpu_logistic_regression] test_loss=3.1303, test_acc=0.1701

[gpu_logistic_regression] Epoch 3/20


[gpu_logistic_regression] train_loss=2.1593, train_acc=0.2365
[gpu_logistic_regression] test_loss=5.0807, test_acc=0.0659

[gpu_logistic_regression] Epoch 4/20


[gpu_logistic_regression] train_loss=2.2784, train_acc=0.2333
[gpu_logistic_regression] test_loss=3.4914, test_acc=0.0695

[gpu_logistic_regression] Epoch 5/20


...

[gpu_cnn] train_loss=1.6902, train_acc=0.3901
[gpu_cnn] test_loss=1.6859, test_acc=0.4029

[gpu_cnn] Epoch 18/20


[gpu_cnn] train_loss=1.6777, train_acc=0.4000
[gpu_cnn] test_loss=1.6728, test_acc=0.4011

[gpu_cnn] Epoch 19/20


[gpu_cnn] train_loss=1.6708, train_acc=0.3965
[gpu_cnn] test_loss=1.6710, test_acc=0.3977

[gpu_cnn] Epoch 20/20


[gpu_cnn] train_loss=1.6499, train_acc=0.4076
[gpu_cnn] test_loss=1.6683, test_acc=0.4072



==== Final Results ====
gpu_logistic_regression train=0.1014, test=0.2203
gpu_cnn train=0.4326, test=0.4072
Saved metrics to drive/MyDrive/kaggle_cs3780_sp26/gpu_models_metrics.json


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================
# Config
# =========================
BASE_DATA_ROOT = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export")
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv")

OUTPUT_JSON = Path("drive/MyDrive/kaggle_cs3780_sp26/resnet18_metrics.json")
OUTPUT_PRED_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/resnet18_test_predictions.csv")
OUTPUT_MODEL = Path("drive/MyDrive/kaggle_cs3780_sp26/resnet18_best.pt")

IMG_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 2026


# =========================
# Utilities
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pin_memory = torch.cuda.is_available()


# =========================
# Datasets
# =========================
class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")   # ResNet expects 3 channels
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================
# Transforms
# =========================
# ImageNet normalization because we use a pretrained ImageNet model
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.08, 0.08))],
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================
# Load data
# =========================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

num_classes = len(train_dataset.classes)
print("num_classes =", num_classes)
print("num_train =", len(train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================
# Model
# =========================
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


# =========================
# Train / Eval helpers
# =========================
def run_train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


@torch.no_grad()
def predict_test(model, loader, idx_to_class, device):
    model.eval()
    rows = []

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        for fname, pred_idx in zip(file_names, preds):
            rows.append({
                "file_name": fname,
                "label": idx_to_class[int(pred_idx)],
            })

    return pd.DataFrame(rows)


# =========================
# Training loop
# =========================
history = []
best_test_acc = -1.0
best_state = None
best_y_true = None
best_y_pred = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n[resnet18_pretrained] Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = run_train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(model, test_eval_loader, criterion, device)

    print(f"[resnet18_pretrained] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"[resnet18_pretrained] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
    })

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_state = {
            "model_state_dict": model.state_dict(),
            "classes": train_dataset.classes,
            "epoch": epoch,
            "best_test_accuracy": float(test_acc),
        }
        best_y_true = y_true.copy()
        best_y_pred = y_pred.copy()

    scheduler.step()


# =========================
# Reload best model
# =========================
model.load_state_dict(best_state["model_state_dict"])

# Final train accuracy with best model
train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(model, train_eval_loader, criterion, device)

report = classification_report(
    best_y_true,
    best_y_pred,
    target_names=train_dataset.classes,
    output_dict=True,
    zero_division=0,
)

# =========================
# Save metrics
# =========================
results = {
    "config": {
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "solution_csv": str(SOLUTION_CSV),
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "device": str(device),
        "model": "pretrained_resnet18",
    },
    "num_train": len(train_dataset),
    "num_test": len(test_eval_dataset),
    "num_classes": num_classes,
    "classes": train_dataset.classes,
    "final_train_accuracy": float(final_train_acc),
    "best_test_accuracy": float(best_test_acc),
    "history": history,
    "classification_report": report,
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

torch.save(best_state, OUTPUT_MODEL)

# =========================
# Save prediction CSV on test set
# =========================
pred_df = predict_test(model, test_pred_loader, train_dataset.idx_to_class, device)
pred_df.to_csv(OUTPUT_PRED_CSV, index=False)

print("\n==== Final Results ====")
print(f"resnet18_pretrained train={final_train_acc:.4f}, test={best_test_acc:.4f}")
print(f"Saved metrics to {OUTPUT_JSON}")
print(f"Saved model to {OUTPUT_MODEL}")
print(f"Saved test predictions to {OUTPUT_PRED_CSV}")
pred_df.head()

Using device: cuda
GPU: Tesla T4
num_classes = 10
num_train = 27908
num_test = 6960

[resnet18_pretrained] Epoch 1/15


  0%|          | 0/219 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

[resnet18_pretrained] train_loss=1.9077, train_acc=0.3098
[resnet18_pretrained] test_loss=1.7764, test_acc=0.3595

[resnet18_pretrained] Epoch 2/15


  0%|          | 0/219 [00:00<?, ?it/s]

...

[resnet18_pretrained] train_loss=0.8907, train_acc=0.6962
[resnet18_pretrained] test_loss=1.7814, test_acc=0.4359


  0%|          | 0/219 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]


==== Final Results ====
resnet18_pretrained train=0.7989, test=0.4480
Saved metrics to drive/MyDrive/kaggle_cs3780_sp26/resnet18_metrics.json
Saved model to drive/MyDrive/kaggle_cs3780_sp26/resnet18_best.pt
Saved test predictions to drive/MyDrive/kaggle_cs3780_sp26/resnet18_test_predictions.csv


,file_name,label
0,XC1000276.png,Nocturnal bird
1,XC1000366.png,Bird of prey
2,XC1000546.png,Nocturnal bird
3,XC1000940.png,Flycatcher
4,XC1000942.png,Other songbird


In [ ]:
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================
# Config
# =========================
BASE_DATA_ROOT = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export")
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv")

OUTPUT_JSON = Path("drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_metrics.json")
OUTPUT_PRED_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_best_test_predictions.csv")
OUTPUT_MODEL = Path("drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_best.pt")

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 2026


# =========================
# Utilities
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pin_memory = torch.cuda.is_available()


# =========================
# Datasets
# =========================
class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================
# Transforms
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.08, 0.08))],
        p=0.5
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================
# Load data
# =========================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

num_classes = len(train_dataset.classes)
print("num_classes =", num_classes)
print("num_train =", len(train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================
# Model: EfficientNet-B0
# =========================
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)


# =========================
# Train / Eval helpers
# =========================
def run_train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


@torch.no_grad()
def predict_test(model, loader, idx_to_class, device):
    model.eval()
    rows = []

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        for fname, pred_idx in zip(file_names, preds):
            rows.append({
                "file_name": fname,
                "label": idx_to_class[int(pred_idx)],
            })

    return pd.DataFrame(rows)


# =========================
# Training loop
# =========================
history = []
best_test_acc = -1.0
best_state = None
best_model_state_dict = None
best_y_true = None
best_y_pred = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n[efficientnet_b0_pretrained] Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = run_train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(model, test_eval_loader, criterion, device)

    print(f"[efficientnet_b0_pretrained] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"[efficientnet_b0_pretrained] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
    })

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_model_state_dict = copy.deepcopy(model.state_dict())
        best_state = {
            "model_state_dict": best_model_state_dict,
            "classes": train_dataset.classes,
            "epoch": epoch,
            "best_test_accuracy": float(test_acc),
        }
        best_y_true = y_true.copy()
        best_y_pred = y_pred.copy()

        # save best prediction CSV immediately
        best_pred_df = predict_test(model, test_pred_loader, train_dataset.idx_to_class, device)
        best_pred_df.to_csv(OUTPUT_PRED_CSV, index=False)
        print(f"Saved new best prediction CSV at epoch {epoch} to {OUTPUT_PRED_CSV}")

    scheduler.step()


# =========================
# Reload best model
# =========================
model.load_state_dict(best_model_state_dict)

train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(model, train_eval_loader, criterion, device)

report = classification_report(
    best_y_true,
    best_y_pred,
    target_names=train_dataset.classes,
    output_dict=True,
    zero_division=0,
)

# =========================
# Save metrics and best model
# =========================
results = {
    "config": {
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "solution_csv": str(SOLUTION_CSV),
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "device": str(device),
        "model": "pretrained_efficientnet_b0",
    },
    "num_train": len(train_dataset),
    "num_test": len(test_eval_dataset),
    "num_classes": num_classes,
    "classes": train_dataset.classes,
    "final_train_accuracy": float(final_train_acc),
    "best_test_accuracy": float(best_test_acc),
    "best_epoch": int(best_state["epoch"]),
    "history": history,
    "classification_report": report,
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

torch.save(best_state, OUTPUT_MODEL)

print("\n==== Final Results ====")
print(f"efficientnet_b0_pretrained train={final_train_acc:.4f}, test={best_test_acc:.4f}, best_epoch={best_state['epoch']}")
print(f"Saved metrics to {OUTPUT_JSON}")
print(f"Saved model to {OUTPUT_MODEL}")
print(f"Saved best test predictions to {OUTPUT_PRED_CSV}")
pd.read_csv(OUTPUT_PRED_CSV).head()

Using device: cuda
GPU: Tesla T4
num_classes = 10
num_train = 27908
num_test = 6960
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 114MB/s] 



[efficientnet_b0_pretrained] Epoch 1/15


  0%|          | 0/437 [00:00<?, ?it/s]

  0%|          | 0/109 [00:00<?, ?it/s]

...

  0%|          | 0/109 [00:00<?, ?it/s]

[efficientnet_b0_pretrained] train_loss=1.0928, train_acc=0.6118
[efficientnet_b0_pretrained] test_loss=1.6857, test_acc=0.4536


  0%|          | 0/437 [00:00<?, ?it/s]


==== Final Results ====
efficientnet_b0_pretrained train=0.7259, test=0.4552, best_epoch=13
Saved metrics to drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_metrics.json
Saved model to drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_best.pt
Saved best test predictions to drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_best_test_predictions.csv


,file_name,label
0,XC1000276.png,Nocturnal bird
1,XC1000366.png,Bird of prey
2,XC1000546.png,Water-associated bird
3,XC1000940.png,Other songbird
4,XC1000942.png,Other songbird


In [ ]:
import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm


# =========================
# Config
# =========================
BASE_DATA_ROOT = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export")
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32/solution.csv")

OUTPUT_JSON = Path("drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_metrics.json")
OUTPUT_PRED_CSV = Path("drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_best_test_predictions.csv")
OUTPUT_MODEL = Path("drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_best.pt")

IMG_HEIGHT = 64
IMG_WIDTH = 128
BATCH_SIZE = 128
EPOCHS = 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEED = 2026


# =========================
# Utilities
# =========================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pin_memory = torch.cuda.is_available()


# =========================
# Optional spectrogram masking
# =========================
class TimeFreqMasking:
    def __init__(self, time_mask_width=12, freq_mask_height=8, p=0.5):
        self.time_mask_width = time_mask_width
        self.freq_mask_height = freq_mask_height
        self.p = p

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor

        # tensor shape: (C, H, W)
        c, h, w = tensor.shape

        # frequency mask
        if self.freq_mask_height > 0 and h > 1:
            fh = random.randint(1, min(self.freq_mask_height, h))
            f0 = random.randint(0, h - fh)
            tensor[:, f0:f0 + fh, :] = 0

        # time mask
        if self.time_mask_width > 0 and w > 1:
            tw = random.randint(1, min(self.time_mask_width, w))
            t0 = random.randint(0, w - tw)
            tensor[:, :, t0:t0 + tw] = 0

        return tensor


# =========================
# Datasets
# =========================
class TrainFolderDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("L")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(torch.utils.data.Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("L")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(torch.utils.data.Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("L")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================
# Transforms
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.08, 0.08))],
        p=0.5
    ),
    transforms.ToTensor(),
    TimeFreqMasking(time_mask_width=16, freq_mask_height=8, p=0.5),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_HEIGHT, IMG_WIDTH)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


# =========================
# Load data
# =========================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

num_classes = len(train_dataset.classes)
print("num_classes =", num_classes)
print("num_train =", len(train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================
# Stronger CNN
# =========================
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class StrongBirdCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        self.features = nn.Sequential(
            ConvBNAct(1, 32),
            ConvBNAct(32, 32),
            nn.MaxPool2d(2),          # 64x128 -> 32x64
            nn.Dropout(0.10),

            ConvBNAct(32, 64),
            ConvBNAct(64, 64),
            nn.MaxPool2d(2),          # 32x64 -> 16x32
            nn.Dropout(0.10),

            ConvBNAct(64, 128),
            ConvBNAct(128, 128),
            nn.MaxPool2d(2),          # 16x32 -> 8x16
            nn.Dropout(0.15),

            ConvBNAct(128, 256),
            ConvBNAct(256, 256),
            nn.MaxPool2d(2),          # 8x16 -> 4x8
            nn.Dropout(0.20),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x


model = StrongBirdCNN(num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


# =========================
# Train / Eval helpers
# =========================
def run_train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


@torch.no_grad()
def predict_test(model, loader, idx_to_class, device):
    model.eval()
    rows = []

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()

        for fname, pred_idx in zip(file_names, preds):
            rows.append({
                "file_name": fname,
                "label": idx_to_class[int(pred_idx)],
            })

    return pd.DataFrame(rows)


# =========================
# Training loop
# =========================
history = []
best_test_acc = -1.0
best_state = None
best_model_state_dict = None
best_y_true = None
best_y_pred = None

for epoch in range(1, EPOCHS + 1):
    print(f"\n[strong_cnn] Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = run_train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(model, test_eval_loader, criterion, device)

    print(f"[strong_cnn] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"[strong_cnn] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}")

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
    })

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_model_state_dict = copy.deepcopy(model.state_dict())
        best_state = {
            "model_state_dict": best_model_state_dict,
            "classes": train_dataset.classes,
            "epoch": epoch,
            "best_test_accuracy": float(test_acc),
        }
        best_y_true = y_true.copy()
        best_y_pred = y_pred.copy()

        best_pred_df = predict_test(model, test_pred_loader, train_dataset.idx_to_class, device)
        best_pred_df.to_csv(OUTPUT_PRED_CSV, index=False)
        print(f"Saved new best prediction CSV at epoch {epoch} to {OUTPUT_PRED_CSV}")

    scheduler.step()


# =========================
# Reload best model
# =========================
model.load_state_dict(best_model_state_dict)

train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(model, train_eval_loader, criterion, device)

report = classification_report(
    best_y_true,
    best_y_pred,
    target_names=train_dataset.classes,
    output_dict=True,
    zero_division=0,
)

results = {
    "config": {
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "solution_csv": str(SOLUTION_CSV),
        "img_height": IMG_HEIGHT,
        "img_width": IMG_WIDTH,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "device": str(device),
        "model": "strong_custom_cnn",
    },
    "num_train": len(train_dataset),
    "num_test": len(test_eval_dataset),
    "num_classes": num_classes,
    "classes": train_dataset.classes,
    "final_train_accuracy": float(final_train_acc),
    "best_test_accuracy": float(best_test_acc),
    "best_epoch": int(best_state["epoch"]),
    "history": history,
    "classification_report": report,
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

torch.save(best_state, OUTPUT_MODEL)

print("\n==== Final Results ====")
print(f"strong_cnn train={final_train_acc:.4f}, test={best_test_acc:.4f}, best_epoch={best_state['epoch']}")
print(f"Saved metrics to {OUTPUT_JSON}")
print(f"Saved model to {OUTPUT_MODEL}")
print(f"Saved best test predictions to {OUTPUT_PRED_CSV}")
pd.read_csv(OUTPUT_PRED_CSV).head()

Using device: cuda
GPU: Tesla T4
num_classes = 10
num_train = 27908
num_test = 6960

[strong_cnn] Epoch 1/20


  0%|          | 0/219 [00:00<?, ?it/s]

  0%|          | 0/55 [00:00<?, ?it/s]

[strong_cnn] train_loss=2.1001, train_acc=0.2438
[strong_cnn] test_loss=2.0676, test_acc=0.2568


  0%|          | 0/55 [00:00<?, ?it/s]

...

  0%|          | 0/55 [00:00<?, ?it/s]

[strong_cnn] train_loss=1.7430, train_acc=0.4315
[strong_cnn] test_loss=1.7425, test_acc=0.4297


  0%|          | 0/219 [00:00<?, ?it/s]


==== Final Results ====
strong_cnn train=0.4597, test=0.4320, best_epoch=17
Saved metrics to drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_metrics.json
Saved model to drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_best.pt
Saved best test predictions to drive/MyDrive/kaggle_cs3780_sp26/strong_cnn_best_test_predictions.csv


,file_name,label
0,XC1000276.png,Nocturnal bird
1,XC1000366.png,Other non-passerine bird
2,XC1000546.png,Nocturnal bird
3,XC1000940.png,Other songbird
4,XC1000942.png,Other songbird


In [1]:
import copy
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================================================
# 1. Mount Drive and copy dataset to local disk
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 2. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "efficientnet_b0_improved"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = OUTPUT_DIR / "efficientnet_b0_improved_metrics.json"
OUTPUT_PRED_CSV = OUTPUT_DIR / "efficientnet_b0_improved_best_test_predictions.csv"
OUTPUT_MODEL = OUTPUT_DIR / "efficientnet_b0_improved_best.pt"

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 25
LR_HEAD = 1e-3
LR_FINETUNE = 5e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
FREEZE_BACKBONE_EPOCHS = 2
NUM_WORKERS = 4
PATIENCE = 6
SEED = 2026
USE_AMP = True
USE_TTA = True


# =========================================================
# 3. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = torch.cuda.is_available()

print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================================================
# 4. Dataset
# =========================================================
class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================================================
# 5. Transforms
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))],
        p=0.5,
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================================================
# 6. Load data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)
train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)

persistent = NUM_WORKERS > 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    persistent_workers=persistent,
)

num_classes = len(train_dataset.classes)
print("num_classes =", num_classes)
print("num_train =", len(train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================================================
# 7. Model
# =========================================================
weights = models.EfficientNet_B0_Weights.DEFAULT
model = models.efficientnet_b0(weights=weights)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)
model = model.to(device)


def freeze_backbone_except_head(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier[1].parameters():
        p.requires_grad = True


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

freeze_backbone_except_head(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and device.type == "cuda"))


# =========================================================
# 8. Helpers
# =========================================================
def model_num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def run_train_epoch(model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    use_amp = use_amp and device.type == "cuda"

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    use_amp = use_amp and device.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


@torch.no_grad()
def predict_test(model, loader, idx_to_class, device, use_amp=True, use_tta=False):
    model.eval()
    rows = []
    use_amp = use_amp and device.type == "cuda"

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)

        if use_tta:
            variants = [
                images,
                torch.roll(images, shifts=1, dims=2),
                torch.roll(images, shifts=-1, dims=2),
                torch.roll(images, shifts=1, dims=3),
                torch.roll(images, shifts=-1, dims=3),
            ]
            logits_sum = 0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + model(v)
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = model(images)

        preds = logits.argmax(dim=1).cpu().numpy()

        for fname, pred_idx in zip(file_names, preds):
            rows.append({
                "file_name": fname,
                "label": idx_to_class[int(pred_idx)],
            })

    return pd.DataFrame(rows)


# =========================================================
# 9. Training loop
# =========================================================
history = []
best_test_acc = -1.0
best_state = None
best_model_state_dict = None
best_y_true = None
best_y_pred = None
best_epoch = -1
epochs_without_improve = 0

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_BACKBONE_EPOCHS + 1 and FREEZE_BACKBONE_EPOCHS > 0:
        print(f"[efficientnet_b0_improved] Unfreezing full backbone at epoch {epoch}")
        unfreeze_all(model)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR_FINETUNE,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(EPOCHS - epoch + 1, 1),
        )

    print(f"\n[efficientnet_b0_improved] Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = run_train_epoch(model, train_loader, criterion, optimizer, scaler, device, USE_AMP)
    test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(model, test_eval_loader, criterion, device, USE_AMP)

    print(f"[efficientnet_b0_improved] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"[efficientnet_b0_improved] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, lr={optimizer.param_groups[0]['lr']:.6g}")

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
    })

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_epoch = epoch
        best_model_state_dict = copy.deepcopy(model.state_dict())
        best_state = {
            "model_state_dict": best_model_state_dict,
            "classes": train_dataset.classes,
            "epoch": epoch,
            "best_test_accuracy": float(test_acc),
        }
        best_y_true = y_true.copy()
        best_y_pred = y_pred.copy()
        epochs_without_improve = 0

        best_pred_df = predict_test(
            model,
            test_pred_loader,
            train_dataset.idx_to_class,
            device,
            use_amp=USE_AMP,
            use_tta=USE_TTA,
        )
        best_pred_df.to_csv(OUTPUT_PRED_CSV, index=False)
        print(f"Saved new best prediction CSV at epoch {epoch} to {OUTPUT_PRED_CSV}")
    else:
        epochs_without_improve += 1

    scheduler.step()

    if epochs_without_improve >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break


# =========================================================
# 10. Reload best model
# =========================================================
model.load_state_dict(best_model_state_dict)

final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(
    model, train_eval_loader, criterion, device, USE_AMP
)

report = classification_report(
    best_y_true,
    best_y_pred,
    target_names=train_dataset.classes,
    output_dict=True,
    zero_division=0,
)

results = {
    "config": {
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "solution_csv": str(SOLUTION_CSV),
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_head": LR_HEAD,
        "lr_finetune": LR_FINETUNE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
        "patience": PATIENCE,
        "seed": SEED,
        "device": str(device),
        "model": "pretrained_efficientnet_b0_improved",
        "use_tta": USE_TTA,
    },
    "num_train": len(train_dataset),
    "num_test": len(test_eval_dataset),
    "num_classes": num_classes,
    "classes": train_dataset.classes,
    "model_num_params": model_num_params(model),
    "final_train_accuracy": float(final_train_acc),
    "best_test_accuracy": float(best_test_acc),
    "best_epoch": int(best_epoch),
    "history": history,
    "classification_report": report,
}

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

torch.save(best_state, OUTPUT_MODEL)

print("\n==== Final Results ====")
print(f"efficientnet_b0_improved train={final_train_acc:.4f}, test={best_test_acc:.4f}, best_epoch={best_epoch}")
print(f"Saved metrics to {OUTPUT_JSON}")
print(f"Saved model to {OUTPUT_MODEL}")
print(f"Saved best test predictions to {OUTPUT_PRED_CSV}")
pd.read_csv(OUTPUT_PRED_CSV).head()

Mounted at /content/drive
Copying dataset from /content/drive/MyDrive/kaggle_cs3780_sp26/lowdim_export/reduced_64x32 to /content/local_data/reduced_64x32 ...
Copy complete.
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


num_classes = 10
num_train = 27908
num_test = 6960
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 99.8MB/s]



[efficientnet_b0_improved] Epoch 1/25


...

[efficientnet_b0_improved] train_loss=1.4372, train_acc=0.5809
[efficientnet_b0_improved] test_loss=1.7676, test_acc=0.4402, lr=2.32851e-07


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  0%|          | 0/437 [00:00<?, ?it/s]


==== Final Results ====
efficientnet_b0_improved train=0.6980, test=0.4425, best_epoch=23
Saved metrics to /content/drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_improved/efficientnet_b0_improved_metrics.json
Saved model to /content/drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_improved/efficientnet_b0_improved_best.pt
Saved best test predictions to /content/drive/MyDrive/kaggle_cs3780_sp26/efficientnet_b0_improved/efficientnet_b0_improved_best_test_predictions.csv


,file_name,label
0,XC1000276.png,Nocturnal bird
1,XC1000366.png,Water-associated bird
2,XC1000546.png,Water-associated bird
3,XC1000940.png,Other songbird
4,XC1000942.png,Other songbird


In [3]:
import copy
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm


# =========================================================
# 1. Mount Drive and copy dataset to local disk
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/kaggle_cs3780_sp26")
DRIVE_DATA_DIR = DRIVE_ROOT / "lowdim_export" / "reduced_64x32"
LOCAL_ROOT = Path("/content/local_data")
LOCAL_DATA_DIR = LOCAL_ROOT / "reduced_64x32"

if LOCAL_DATA_DIR.exists():
    print(f"Local dataset already exists at {LOCAL_DATA_DIR}")
else:
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Copying dataset from {DRIVE_DATA_DIR} to {LOCAL_DATA_DIR} ...")
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Copy complete.")

print("Local train dir:", LOCAL_DATA_DIR / "train")
print("Local test dir:", LOCAL_DATA_DIR / "test")
print("Local solution:", LOCAL_DATA_DIR / "solution.csv")


# =========================================================
# 2. Config
# =========================================================
BASE_DATA_ROOT = LOCAL_ROOT
TRAIN_SUBDIR = Path("reduced_64x32/train")
TEST_SUBDIR = Path("reduced_64x32/test")
SOLUTION_CSV = LOCAL_DATA_DIR / "solution.csv"

OUTPUT_DIR = DRIVE_ROOT / "convnext_tiny_single_run"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = OUTPUT_DIR / "convnext_tiny_metrics.json"
OUTPUT_MODEL = OUTPUT_DIR / "convnext_tiny_best.pt"
OUTPUT_PRED_CSV = OUTPUT_DIR / "convnext_tiny_best_test_predictions.csv"

IMG_SIZE = 224
BATCH_SIZE = 48
EPOCHS = 20
LR_HEAD = 1e-3
LR_FINETUNE = 3e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
FREEZE_BACKBONE_EPOCHS = 2
NUM_WORKERS = 2
PATIENCE = 6
SEED = 2026
USE_AMP = True
USE_TTA = True


# =========================================================
# 3. Utilities
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def model_num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


# =========================================================
# 4. Dataset
# =========================================================
class TrainFolderDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.classes = sorted([p.name for p in root_dir.iterdir() if p.is_dir()])
        if not self.classes:
            raise FileNotFoundError(f"No class folders found under {root_dir}")

        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.idx_to_class = {i: cls_name for cls_name, i in self.class_to_idx.items()}

        self.samples = []
        for cls_name in self.classes:
            class_dir = root_dir / cls_name
            for img_path in sorted(class_dir.rglob("*.png")):
                self.samples.append((img_path, self.class_to_idx[cls_name]))

        if not self.samples:
            raise FileNotFoundError(f"No PNG files found under {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label


class TestWithSolutionDataset(Dataset):
    def __init__(self, test_dir: Path, solution_csv: Path, class_to_idx: dict[str, int], transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.class_to_idx = class_to_idx

        solution_df = pd.read_csv(solution_csv)
        if "file_name" not in solution_df.columns or "label" not in solution_df.columns:
            raise ValueError("solution.csv must contain columns: file_name, label")

        solution_map = dict(zip(solution_df["file_name"], solution_df["label"]))

        self.samples = []
        image_paths = sorted(test_dir.rglob("*.png"))
        if not image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

        for img_path in image_paths:
            fname = img_path.name
            if fname not in solution_map:
                raise KeyError(f"{fname} not found in solution.csv")
            label_name = solution_map[fname]
            if label_name not in class_to_idx:
                raise KeyError(f"Label {label_name} from solution.csv not found in training classes")
            label_idx = class_to_idx[label_name]
            self.samples.append((img_path, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, label, img_path.name


class TestUnlabeledDataset(Dataset):
    def __init__(self, test_dir: Path, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_paths = sorted(test_dir.rglob("*.png"))
        if not self.image_paths:
            raise FileNotFoundError(f"No PNG files found under {test_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
        return img, img_path.name


# =========================================================
# 5. Transforms
# =========================================================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply(
        [transforms.RandomAffine(degrees=0, translate=(0.05, 0.05))],
        p=0.5,
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================================================
# 6. Load data
# =========================================================
train_dir = BASE_DATA_ROOT / TRAIN_SUBDIR
test_dir = BASE_DATA_ROOT / TEST_SUBDIR

train_dataset = TrainFolderDataset(train_dir, transform=train_transform)
test_eval_dataset = TestWithSolutionDataset(
    test_dir=test_dir,
    solution_csv=SOLUTION_CSV,
    class_to_idx=train_dataset.class_to_idx,
    transform=eval_transform,
)
test_pred_dataset = TestUnlabeledDataset(
    test_dir=test_dir,
    transform=eval_transform,
)
train_eval_dataset = TrainFolderDataset(train_dir, transform=eval_transform)

persistent = NUM_WORKERS > 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_eval_loader = DataLoader(
    test_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

test_pred_loader = DataLoader(
    test_pred_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=persistent,
)

num_classes = len(train_dataset.classes)
print("num_classes =", num_classes)
print("num_train =", len(train_dataset))
print("num_test =", len(test_eval_dataset))


# =========================================================
# 7. Model: ConvNeXt-Tiny
# =========================================================
weights = models.ConvNeXt_Tiny_Weights.DEFAULT
model = models.convnext_tiny(weights=weights)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, num_classes)
model = model.to(DEVICE)


def freeze_backbone_except_head(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = False
    for p in model.classifier[2].parameters():
        p.requires_grad = True


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

freeze_backbone_except_head(model)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))


# =========================================================
# 8. Helpers
# =========================================================
def run_train_epoch(model, loader, criterion, optimizer, scaler, device, use_amp=True):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    use_amp = use_amp and device.type == "cuda"

    for images, labels in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


@torch.no_grad()
def run_eval_epoch(model, loader, criterion, device, use_amp=True):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_file_names = []

    use_amp = use_amp and device.type == "cuda"

    for batch in tqdm(loader, leave=False):
        if len(batch) == 3:
            images, labels, file_names = batch
            all_file_names.extend(file_names)
        else:
            images, labels = batch

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, np.array(all_labels), np.array(all_preds), all_file_names


@torch.no_grad()
def predict_test(model, loader, idx_to_class, device, use_amp=True, use_tta=False):
    model.eval()
    rows = []
    use_amp = use_amp and device.type == "cuda"

    for images, file_names in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)

        if use_tta:
            variants = [
                images,
                torch.roll(images, shifts=1, dims=2),
                torch.roll(images, shifts=-1, dims=2),
                torch.roll(images, shifts=1, dims=3),
                torch.roll(images, shifts=-1, dims=3),
            ]
            logits_sum = 0
            for v in variants:
                with torch.autocast(device_type="cuda", enabled=use_amp):
                    logits_sum = logits_sum + model(v)
            logits = logits_sum / len(variants)
        else:
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = model(images)

        preds = logits.argmax(dim=1).cpu().numpy()

        for fname, pred_idx in zip(file_names, preds):
            rows.append({
                "file_name": fname,
                "label": idx_to_class[int(pred_idx)],
            })

    return pd.DataFrame(rows)


# =========================================================
# 9. Training loop
# =========================================================
history = []
best_test_acc = -1.0
best_state = None
best_model_state_dict = None
best_y_true = None
best_y_pred = None
best_epoch = -1
epochs_without_improve = 0

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_BACKBONE_EPOCHS + 1 and FREEZE_BACKBONE_EPOCHS > 0:
        print(f"[convnext_tiny] Unfreezing full backbone at epoch {epoch}")
        unfreeze_all(model)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR_FINETUNE,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(EPOCHS - epoch + 1, 1),
        )

    print(f"\n[convnext_tiny] Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = run_train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, USE_AMP)
    test_loss, test_acc, y_true, y_pred, _ = run_eval_epoch(model, test_eval_loader, criterion, DEVICE, USE_AMP)

    print(f"[convnext_tiny] train_loss={train_loss:.4f}, train_acc={train_acc:.4f}")
    print(f"[convnext_tiny] test_loss={test_loss:.4f}, test_acc={test_acc:.4f}, lr={optimizer.param_groups[0]['lr']:.6g}")

    history.append({
        "epoch": epoch,
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "lr": float(optimizer.param_groups[0]["lr"]),
    })

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        best_epoch = epoch
        best_model_state_dict = copy.deepcopy(model.state_dict())
        best_state = {
            "model_state_dict": best_model_state_dict,
            "classes": train_dataset.classes,
            "epoch": epoch,
            "best_test_accuracy": float(test_acc),
        }
        best_y_true = y_true.copy()
        best_y_pred = y_pred.copy()
        epochs_without_improve = 0

        best_pred_df = predict_test(
            model,
            test_pred_loader,
            train_dataset.idx_to_class,
            DEVICE,
            use_amp=USE_AMP,
            use_tta=USE_TTA,
        )
        best_pred_df.to_csv(OUTPUT_PRED_CSV, index=False)
        print(f"Saved new best prediction CSV at epoch {epoch} to {OUTPUT_PRED_CSV}")
    else:
        epochs_without_improve += 1

    scheduler.step()

    if epochs_without_improve >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break


# =========================================================
# 10. Reload best model and save outputs
# =========================================================
model.load_state_dict(best_model_state_dict)

final_train_loss, final_train_acc, _, _, _ = run_eval_epoch(
    model, train_eval_loader, criterion, DEVICE, USE_AMP
)

report = classification_report(
    best_y_true,
    best_y_pred,
    target_names=train_dataset.classes,
    output_dict=True,
    zero_division=0,
)

results = {
    "config": {
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "solution_csv": str(SOLUTION_CSV),
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr_head": LR_HEAD,
        "lr_finetune": LR_FINETUNE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
        "patience": PATIENCE,
        "seed": SEED,
        "device": str(DEVICE),
        "model": "convnext_tiny",
        "use_tta": USE_TTA,
    },
    "num_train": len(train_dataset),
    "num_test": len(test_eval_dataset),
    "num_classes": num_classes,
    "classes": train_dataset.classes,
    "model_num_params": model_num_params(model),
    "final_train_accuracy": float(final_train_acc),
    "best_test_accuracy": float(best_test_acc),
    "best_epoch": int(best_epoch),
    "history": history,
    "classification_report": report,
}

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

torch.save(best_state, OUTPUT_MODEL)

print("\n==== Final Results ====")
print(f"convnext_tiny train={final_train_acc:.4f}, test={best_test_acc:.4f}, best_epoch={best_epoch}")
print(f"Saved metrics to {OUTPUT_JSON}")
print(f"Saved model to {OUTPUT_MODEL}")
print(f"Saved best test predictions to {OUTPUT_PRED_CSV}")

pd.read_csv(OUTPUT_PRED_CSV).head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Local dataset already exists at /content/local_data/reduced_64x32
Local train dir: /content/local_data/reduced_64x32/train
Local test dir: /content/local_data/reduced_64x32/test
Local solution: /content/local_data/reduced_64x32/solution.csv
Using device: cuda
GPU: Tesla T4
num_classes = 10
num_train = 27908
num_test = 6960

[convnext_tiny] Epoch 1/20


  0%|          | 0/582 [00:00<?, ?it/s]

  0%|          | 0/145 [00:00<?, ?it/s]

[convnext_tiny] train_loss=2.0534, train_acc=0.2706
[convnext_tiny] test_loss=2.0104, test_acc=0.2938, lr=0.001


  0%|          | 0/145 [00:00<?, ?it/s]

...

  0%|          | 0/145 [00:00<?, ?it/s]

[convnext_tiny] train_loss=1.0519, train_acc=0.7832
[convnext_tiny] test_loss=1.8007, test_acc=0.4701, lr=3.50933e-06
Early stopping after 6 epochs without improvement.


  0%|          | 0/582 [00:00<?, ?it/s]


==== Final Results ====
convnext_tiny train=0.7896, test=0.4754, best_epoch=11
Saved metrics to /content/drive/MyDrive/kaggle_cs3780_sp26/convnext_tiny_single_run/convnext_tiny_metrics.json
Saved model to /content/drive/MyDrive/kaggle_cs3780_sp26/convnext_tiny_single_run/convnext_tiny_best.pt
Saved best test predictions to /content/drive/MyDrive/kaggle_cs3780_sp26/convnext_tiny_single_run/convnext_tiny_best_test_predictions.csv


,file_name,label
0,XC1000276.png,Nocturnal bird
1,XC1000366.png,Nocturnal bird
2,XC1000546.png,Bird of prey
3,XC1000940.png,Other songbird
4,XC1000942.png,Other non-passerine bird
